In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np


In [3]:
Tickers = ["CDSL.NS","RELIANCE.NS","TCS.NS","HDFCBANK.NS"]

data = yf.download(Tickers,period="1y")["Close"]
print(data.head())

[*********************100%***********************]  4 of 4 completed

Ticker          CDSL.NS  HDFCBANK.NS  RELIANCE.NS       TCS.NS
2025-08-28  1438.874878   942.216248  1379.521729  2970.390625
2025-08-29  1411.524414   936.117126  1350.953735  2961.749268
2025-09-01  1463.549805   935.133362  1347.668945  2988.537354
2025-09-02  1495.260620   929.132690  1360.210938  2987.481201
2025-09-03  1504.872803   938.920776  1366.282837  2974.039307


In [4]:
returns = data.pct_change().dropna()
print(returns.head())

Ticker       CDSL.NS  HDFCBANK.NS  RELIANCE.NS    TCS.NS
2025-08-29 -0.019008    -0.006473    -0.020709 -0.002909
2025-09-01  0.036858    -0.001051    -0.002431  0.009045
2025-09-02  0.021667    -0.006417     0.009306 -0.000353
2025-09-03  0.006428     0.010535     0.004464 -0.004499
2025-09-04 -0.006058     0.007124    -0.009690 -0.000581


In [5]:
weights = [0.25, 0.25, 0.25, 0.25]

portfolio_returns = (returns * weights).sum(axis=1)
print(portfolio_returns.head())

2025-08-29   -0.012275
2025-09-01    0.010605
2025-09-02    0.006051
2025-09-03    0.004232
2025-09-04   -0.002301
dtype: float64


In [6]:
var_95 = -portfolio_returns.quantile(0.05)
print("VaR (95%):", var_95)

VaR (95%): 0.019589263574894114


In [7]:
risk_free_rate = 0.06  # rough Indian risk-free proxy
excess_daily_return = portfolio_returns.mean() - (risk_free_rate / 252)
sharpe_ratio = (excess_daily_return / portfolio_returns.std()) * np.sqrt(252)
print("Sharpe Ratio:", sharpe_ratio)

Sharpe Ratio: -0.9932769718414624


In [8]:
cumulative = (1 + portfolio_returns).cumprod()
running_max = cumulative.cummax()
drawdown = (cumulative - running_max) / running_max
max_drawdown = drawdown.min()
print("Max Drawdown:", max_drawdown)

Max Drawdown: -0.24811126238630227


In [9]:
correlation_matrix = returns.corr()
print(correlation_matrix)

Ticker        CDSL.NS  HDFCBANK.NS  RELIANCE.NS    TCS.NS
Ticker                                                   
CDSL.NS      1.000000     0.539407     0.433494  0.269817
HDFCBANK.NS  0.539407     1.000000     0.415356  0.219357
RELIANCE.NS  0.433494     0.415356     1.000000  0.161224
TCS.NS       0.269817     0.219357     0.161224  1.000000


In [10]:
def compute_risk_snapshot(tickers, weights, period="1y", risk_free_rate=0.06):
    data = yf.download(tickers, period=period)["Close"]
    returns = data.pct_change().dropna()
    portfolio_returns = (returns * weights).sum(axis=1)

    var_95 = -portfolio_returns.quantile(0.05)

    excess_daily_return = portfolio_returns.mean() - (risk_free_rate / 252)
    sharpe = (excess_daily_return / portfolio_returns.std()) * np.sqrt(252)

    cumulative = (1 + portfolio_returns).cumprod()
    running_max = cumulative.cummax()
    max_drawdown = ((cumulative - running_max) / running_max).min()

    correlation_matrix = returns.corr()

    return {
        "var_95": var_95,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
        "correlation_matrix": correlation_matrix.to_dict()
    }

In [11]:
snapshot = compute_risk_snapshot(
    tickers=["CDSL.NS", "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS"],
    weights=[0.25, 0.25, 0.25, 0.25]
)
print(snapshot)

[*********************100%***********************]  4 of 4 completed

{'var_95': np.float64(0.019589273183567443), 'sharpe': np.float64(-0.9932769613457562), 'max_drawdown': np.float64(-0.2481112620828382), 'correlation_matrix': {'CDSL.NS': {'CDSL.NS': 1.0, 'HDFCBANK.NS': 0.5394074565856783, 'RELIANCE.NS': 0.4334938076544893, 'TCS.NS': 0.269816750244235}, 'HDFCBANK.NS': {'CDSL.NS': 0.5394074565856783, 'HDFCBANK.NS': 1.0, 'RELIANCE.NS': 0.41535615634809864, 'TCS.NS': 0.21935648573390168}, 'RELIANCE.NS': {'CDSL.NS': 0.4334938076544893, 'HDFCBANK.NS': 0.41535615634809864, 'RELIANCE.NS': 1.0, 'TCS.NS': 0.1612235296739912}, 'TCS.NS': {'CDSL.NS': 0.269816750244235, 'HDFCBANK.NS': 0.21935648573390168, 'RELIANCE.NS': 0.1612235296739912, 'TCS.NS': 1.0}}}
